# Notebook 02 — Descriptive Statistics and Distributions

**Module:** 03 — Statistics and Probability  
**Notebook:** 02 of 18  
**Prerequisites:** NB01  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Before running a single statistical test, you must understand your data. Descriptive
statistics are the vocabulary of data exploration: they summarise shape, centre,
spread, and outliers. Choosing the wrong summary statistic — reporting the mean
of a skewed distribution, or ignoring bimodality — is a common, avoidable error
in published genomics papers.

---
## Step 2 — Intuition

Think of descriptive statistics as answering four questions about any dataset:

1. **Where is the middle?** (mean, median, mode)
2. **How spread out is it?** (variance, standard deviation, IQR, range)
3. **What shape does it have?** (symmetric, skewed, bimodal — histogram, KDE)
4. **Are there unusual values?** (outliers — box plot, z-score)

The answers determine which statistical tests are valid.

---
## Step 3 — Biological Background

**Why raw gene expression counts are skewed:**
- Most genes are unexpressed or lowly expressed: many near-zero counts
- A small number of housekeeping genes have very high expression
- The resulting distribution is right-skewed, heavy-tailed
- **Standard practice:** log₂-transform before any parametric statistics

**GC content distribution:**
- GC content across genomic windows follows a roughly normal distribution
- Deviations (bimodal GC) can indicate contamination or misassembly
- Standard QC step in any genome assembly or metagenomics project

**Coefficient of variation (CV):**
$$\text{CV} = \sigma / \mu$$
Used to compare variability across genes with very different mean expressions —
a gene with CV=0.5 is more variable relative to its mean than one with CV=0.1.

---
## Step 4 — Mathematical Explanation

**Measures of centre:**
$$\text{mean}: \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$
$$\text{median}: \text{middle value when sorted (robust to outliers)}$$

**Measures of spread:**
$$\text{variance}: s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2$$
The $n-1$ (Bessel's correction) makes $s^2$ an unbiased estimator of $\sigma^2$.
$$\text{IQR} = Q_3 - Q_1$$

**Outlier detection (IQR rule):**
$$\text{outlier if } x < Q_1 - 1.5 \cdot \text{IQR} \text{ or } x > Q_3 + 1.5 \cdot \text{IQR}$$

**Skewness:** positive skew = right tail; negative skew = left tail.
RNA-seq raw counts: positive skew → use log transform.

---
## Step 5 — Computational Explanation

Key NumPy/SciPy functions:

| Quantity | Function |
|----------|----------|
| Mean | `np.mean(x)` |
| Median | `np.median(x)` |
| Standard deviation | `np.std(x, ddof=1)` (use ddof=1 for sample SD) |
| Variance | `np.var(x, ddof=1)` |
| Percentiles | `np.percentile(x, [25, 50, 75])` |
| Skewness, kurtosis | `scipy.stats.skew(x)`, `scipy.stats.kurtosis(x)` |
| All-in-one summary | `scipy.stats.describe(x)` |

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# Cell 6.1 — Implement descriptive stats from scratch, then validate against numpy
def descriptive_stats(x: np.ndarray) -> dict:
    """
    Compute key descriptive statistics from scratch.

    Returns a dict with: n, mean, median, std, variance, iqr, skewness, min, max
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = x.sum() / n
    # Sample variance (Bessel's correction: n-1)
    variance = np.sum((x - mean)**2) / (n - 1)
    std = np.sqrt(variance)
    x_sorted = np.sort(x)
    median = np.median(x)  # use numpy for edge cases with even n
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    # Skewness (Fisher's)
    skewness = (np.sum((x - mean)**3) / n) / std**3
    return dict(n=n, mean=mean, median=median, std=std,
                variance=variance, iqr=iqr, skewness=skewness,
                min=x_sorted[0], max=x_sorted[-1])

# Validate against scipy
rng = np.random.default_rng(42)
x_test = rng.lognormal(mean=2, sigma=0.8, size=200)  # right-skewed like raw counts

my_stats = descriptive_stats(x_test)
scipy_desc = stats.describe(x_test)

print("Custom vs scipy:")
print(f"  mean:     {my_stats['mean']:.4f}  vs  {scipy_desc.mean:.4f}")
print(f"  variance: {my_stats['variance']:.4f}  vs  {scipy_desc.variance:.4f}")
print(f"  skewness: {my_stats['skewness']:.4f}  vs  {scipy_desc.skewness:.4f}")
print(f"\nDistribution shape: skewness = {my_stats['skewness']:.2f} → right-skewed (expected for lognormal)")

In [ ]:
# Cell 6.2 — Gene expression matrix: synthetic, 500 genes × 20 samples
rng = np.random.default_rng(42)
n_genes, n_samples = 500, 20
groups = ['control'] * 10 + ['treatment'] * 10

# Raw counts: Negative Binomial-like (overdispersed Poisson)
# Simplified: use exponential + Poisson noise
true_expr = rng.lognormal(mean=3, sigma=2, size=n_genes)  # gene mean expressions
raw_counts = rng.poisson(lam=true_expr[:, np.newaxis] * np.ones(n_samples))

# Log2-CPM normalization
library_sizes = raw_counts.sum(axis=0)  # per-sample library size
cpm = raw_counts / library_sizes[np.newaxis, :] * 1e6
log2cpm = np.log2(cpm + 1)  # +1 pseudo-count avoids log(0)

print(f"Expression matrix: {n_genes} genes × {n_samples} samples")
print(f"Raw counts — mean: {raw_counts.mean():.1f}, max: {raw_counts.max()}, skewness: {stats.skew(raw_counts.ravel()):.2f}")
print(f"log2CPM   — mean: {log2cpm.mean():.2f},  max: {log2cpm.max():.2f}, skewness: {stats.skew(log2cpm.ravel()):.2f}")

In [ ]:
# Cell 6.3 — GC content distribution across genes
# Simulate GC content per gene (roughly normal, mean ~50%, std ~8%)
gc_content = rng.normal(loc=0.50, scale=0.08, size=n_genes)
gc_content = np.clip(gc_content, 0, 1)  # GC must be in [0,1]

gc_stats = descriptive_stats(gc_content)
Q1, Q3 = np.percentile(gc_content, [25, 75])
IQR = Q3 - Q1
outlier_mask = (gc_content < Q1 - 1.5*IQR) | (gc_content > Q3 + 1.5*IQR)

print(f"GC content — mean: {gc_stats['mean']:.3f}, std: {gc_stats['std']:.3f}, IQR: {gc_stats['iqr']:.3f}")
print(f"Outliers (IQR rule): {outlier_mask.sum()} genes")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Four panels: raw vs. log, histogram, box plot, violin
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Panel 1: raw count distribution (skewed)
ax = axes[0, 0]
ax.hist(raw_counts.ravel(), bins=60, color="steelblue", edgecolor="none", density=True)
ax.set_xlabel("Raw count"); ax.set_ylabel("Density")
ax.set_title(f"Raw counts (skewness = {stats.skew(raw_counts.ravel()):.1f})")
ax.set_xlim(0, np.percentile(raw_counts, 99))

# Panel 2: log2CPM distribution (approximately normal)
ax = axes[0, 1]
ax.hist(log2cpm.ravel(), bins=50, color="green", edgecolor="none", density=True)
x_fit = np.linspace(log2cpm.min(), log2cpm.max(), 300)
ax.plot(x_fit, stats.norm.pdf(x_fit, log2cpm.mean(), log2cpm.std()), 'k-', lw=1.5, label="Normal fit")
ax.set_xlabel("log₂CPM"); ax.set_ylabel("Density")
ax.set_title(f"log₂CPM (skewness = {stats.skew(log2cpm.ravel()):.2f})")
ax.legend()

# Panel 3: box plot across samples
ax = axes[1, 0]
ax.boxplot(log2cpm, vert=True, patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.5),
           medianprops=dict(color="black"))
ax.axhline(log2cpm.mean(), color="tomato", ls="--", lw=1, label="Grand mean")
ax.set_xlabel("Sample"); ax.set_ylabel("log₂CPM")
ax.set_title("Box plot: expression per sample (QC view)")
ax.set_xticks(range(1, n_samples+1, 5))
ax.legend()

# Panel 4: GC content with outlier annotation
ax = axes[1, 1]
ax.hist(gc_content[~outlier_mask], bins=30, color="steelblue", edgecolor="none", density=True, label="Normal")
ax.scatter(gc_content[outlier_mask], np.zeros(outlier_mask.sum()) + 0.5,
           color="tomato", s=40, zorder=5, label=f"Outliers (n={outlier_mask.sum()})")
ax.set_xlabel("GC content"); ax.set_ylabel("Density")
ax.set_title("GC content distribution")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `descriptive_stats()` from scratch without looking at Cell 6.1. Verify it
   gives the same result for a vector of your choice.
2. Add `coefficient_of_variation` to `descriptive_stats()` and compute it for both
   the raw counts and log2CPM distributions. Interpret the difference.
3. The IQR outlier rule declares values beyond $Q_1 \pm 1.5 \cdot \text{IQR}$ as outliers.
   For a Normal distribution, what fraction of observations does this flag? Compute analytically
   using `scipy.stats.norm.cdf`, then verify by simulation.
4. The box plot in Panel 3 shows per-sample medians roughly aligned. If one sample's
   median were shifted by 2 log2CPM units, what would that indicate biologically?
   How would you detect and handle it?

---
## Quiz — Active Recall

1. Why use $n-1$ (Bessel's correction) in the sample variance formula?
2. When is the median a better measure of centre than the mean? Give a biological example.
3. What does positive skewness look like on a histogram? What causes it in RNA-seq data?
4. Write the IQR outlier detection rule from memory.
5. Why do we log-transform expression counts before running t-tests?

---
## Reflection

**Date completed:** ____________________

> *[Can you explain why raw RNA-seq counts are skewed and what log-transformation achieves?]*

---
*Next: `03_probability_fundamentals.ipynb`*